In [1]:
!nvidia-smi

Tue Nov 10 01:50:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
import pandas as pd
import torch
import transformers
import numpy as np
from transformers import AutoTokenizer
import torch
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer, TrainingArguments


In [4]:

train = pd.read_csv('train.csv')
val = pd.read_csv('val.csv')
test = pd.read_csv('test.csv')

print(train.shape, val.shape, test.shape)
print('GPU available:', torch.cuda.is_available())
print('transformers version:', transformers.__version__)

(9811, 5) (2103, 5) (2103, 5)
GPU available: True
transformers version: 5.22.1


In [5]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = train['text_clean'][0]
print(example)
print(tokenizer.tokenize(example))
print(tokenizer(example)['input_ids'])

ids = tokenizer(train['text_clean'].tolist())['input_ids']
lengths = np.array([len(x) for x in ids])
print('longest:', lengths.max())
print('95th percentile:', int(np.percentile(lengths, 95)))
print('99th percentile:', int(np.percentile(lengths, 99)))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Wife and I flew to Italy recently. Bought tickets via Finnair, flights operated by USAir. Didn't get dividend miles credited.
['wife', 'and', 'i', 'flew', 'to', 'italy', 'recently', '.', 'bought', 'tickets', 'via', 'finn', '##air', ',', 'flights', 'operated', 'by', 'usa', '##ir', '.', 'didn', "'", 't', 'get', 'divide', '##nd', 'miles', 'credited', '.']
[101, 2564, 1998, 1045, 5520, 2000, 3304, 3728, 1012, 4149, 9735, 3081, 9303, 11215, 1010, 7599, 3498, 2011, 3915, 4313, 1012, 2134, 1005, 1056, 2131, 11443, 4859, 2661, 5827, 1012, 102]
longest: 52
95th percentile: 36
99th percentile: 40


In [8]:
MAX_LEN = 64

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, max_length=MAX_LEN, return_token_type_ids=False)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item['labels'] = int(self.labels[i])
        return item

train_ds = TweetDataset(train['text_clean'].tolist(), train['label'].tolist())
val_ds = TweetDataset(val['text_clean'].tolist(), val['label'].tolist())
test_ds = TweetDataset(test['text_clean'].tolist(), test['label'].tolist())

collator = DataCollatorWithPadding(tokenizer)

print(len(train_ds), len(val_ds), len(test_ds))
print(train_ds[0])

9811 2103 2103
{'input_ids': [101, 2564, 1998, 1045, 5520, 2000, 3304, 3728, 1012, 4149, 9735, 3081, 9303, 11215, 1010, 7599, 3498, 2011, 3915, 4313, 1012, 2134, 1005, 1056, 2131, 11443, 4859, 2661, 5827, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0}


In [9]:
print(train_ds[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [11]:


model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to('cuda')
print(round(sum(p.numel() for p in model.parameters()) / 1e6, 1), 'million parameters')

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


67.0 million parameters


In [13]:
classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=train['label'].values)
print(dict(zip(classes.tolist(), weights.round(2))))
class_weights = torch.tensor(weights, dtype=torch.float).to('cuda')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'accuracy': float((preds == labels).mean()),
    }

{0: np.float64(0.52), 1: np.float64(1.63), 2: np.float64(2.17)}


In [16]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=class_weights)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir='distilbert_out',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=92,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=True,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
print('trainer ready')

trainer ready


In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.489564,0.552742,0.707443,0.737993
2,0.393782,0.521194,0.788487,0.833571
3,0.331901,0.541492,0.786920,0.831194


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=921, training_loss=0.4630340779125367, metrics={'train_runtime': 76.3915, 'train_samples_per_second': 385.291, 'train_steps_per_second': 12.056, 'total_flos': 297518027709834.0, 'train_loss': 0.4630340779125367, 'epoch': 3.0})

In [18]:
from sklearn.metrics import classification_report

val_out = trainer.predict(val_ds)
val_pred = val_out.predictions.argmax(axis=-1)
print(classification_report(val['label'], val_pred, target_names=['negative', 'neutral', 'positive']))

              precision    recall  f1-score   support

    negative       0.92      0.88      0.90      1351
     neutral       0.64      0.71      0.68       429
    positive       0.77      0.82      0.79       323

    accuracy                           0.83      2103
   macro avg       0.78      0.80      0.79      2103
weighted avg       0.84      0.83      0.84      2103



In [19]:
test_out = trainer.predict(test_ds)
test_pred = test_out.predictions.argmax(axis=-1)

print('test macro-F1:', round(f1_score(test['label'], test_pred, average='macro'), 3))
print(classification_report(test['label'], test_pred, target_names=['negative', 'neutral', 'positive']))

test macro-F1: 0.802
              precision    recall  f1-score   support

    negative       0.94      0.88      0.91      1351
     neutral       0.65      0.76      0.70       430
    positive       0.77      0.83      0.80       322

    accuracy                           0.84      2103
   macro avg       0.79      0.82      0.80      2103
weighted avg       0.85      0.84      0.85      2103



In [20]:
model.config.id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
model.config.label2id = {'negative': 0, 'neutral': 1, 'positive': 2}

trainer.save_model('distilbert_final')
tokenizer.save_pretrained('distilbert_final')

import shutil
shutil.make_archive('distilbert_final', 'zip', 'distilbert_final')

from google.colab import files
files.download('distilbert_final.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>